In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

At first glance, it seemed to me that the model had invented this path to reach the end result, where there might be multiple ways of doing things — but it's all about the model finding the most effective path along the gradient, and in our case, this happened to be the one it found.

In [8]:
import numpy as np

def trig_identity_add(a: int, b: int, p: int, freq: int = 1) -> int:
    """
    Computes a + b mod p using ONLY the trig identity -- no neural network,
    no embeddings, just cos/sin. This is the pure math the network's circuit
    is equivalent to.
    """
    # place a and b on a circle as angles
    theta_a = 2 * np.pi * freq * a / p
    theta_b = 2 * np.pi * freq * b / p

    # combine the two angles using the angle-addition identity
    # cos(theta_a + theta_b) = cos(theta_a)cos(theta_b) - sin(theta_a)sin(theta_b)
    # sin(theta_a + theta_b) = sin(theta_a)cos(theta_b) + cos(theta_a)sin(theta_b)
    cos_sum = np.cos(theta_a) * np.cos(theta_b) - np.sin(theta_a) * np.sin(theta_b)
    sin_sum = np.sin(theta_a) * np.cos(theta_b) + np.cos(theta_a) * np.sin(theta_b)

    # try every possible answer c, score how well it matches
    # the combined angle using cos(theta_sum - theta_c)
    best_c, best_score = None, -np.inf
    for c in range(p):
        theta_c = 2 * np.pi * freq * c / p
        score = cos_sum * np.cos(theta_c) + sin_sum * np.sin(theta_c)
        if score > best_score:
            best_score = score
            best_c = c

    return best_c


# quick test against ordinary modular addition
p = 113
for _ in range(10):
    a, b = np.random.randint(0, p, size=2)
    predicted = trig_identity_add(a, b, p)
    actual = (a + b) % p
    print(f"a={a:3d} b={b:3d}  trig_predicted={predicted:3d}  actual={actual:3d}  match={predicted == actual}")

a= 68 b= 40  trig_predicted=108  actual=108  match=True
a= 81 b=  3  trig_predicted= 84  actual= 84  match=True
a= 75 b= 82  trig_predicted= 44  actual= 44  match=True
a= 63 b= 30  trig_predicted= 93  actual= 93  match=True
a= 59 b= 94  trig_predicted= 40  actual= 40  match=True
a=  5 b= 38  trig_predicted= 43  actual= 43  match=True
a=110 b=107  trig_predicted=104  actual=104  match=True
a= 56 b= 10  trig_predicted= 66  actual= 66  match=True
a=103 b= 89  trig_predicted= 79  actual= 79  match=True
a= 64 b=109  trig_predicted= 60  actual= 60  match=True


### Change in the phase transition with respect to the decay parameter λ

In [25]:
import os
from pathlib import Path
from mintrans_clean import *



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

l = [0.07, 0.08, 0.09, 1]

checkpoint_dir = './skeletons'
if not os.path.exists(checkpoint_dir): 
    os.makedirs(checkpoint_dir)
# general_paramater in path refers to the configuration in minitrans_clean.py with different weight decay



# ---------------------------------------------------------------------------
# General configuration
# ---------------------------------------------------------------------------

cfg = TrainConfig

# ---------------------------------------------------------------------------
# Data loader configuration
# ---------------------------------------------------------------------------

torch.manual_seed(cfg.torch_seed)

train_ds = FibonacciTrainDataset(
    mod=cfg.p, seq_len=cfg.seq_len, train_frac=cfg.train_frac,
    seed=cfg.data_seed,
)
val_ds = FibonacciValDataset(train_ds, num_samples=500)

n_unseen = sum(
    1 for a in range(cfg.p) for b in range(cfg.p)
    if (a, b) not in train_ds.seen_pairs
)

cfg.batch_size = len(train_ds)   
cfg.epochs = 100

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

# ---------------------------------------------------------------------------
# Model configuration 
# ---------------------------------------------------------------------------

model = MinimalTransformer(
    vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
    num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
    hidden_mlp=cfg.hidden_mlp,
).to(device)


for i in range(len(l)):
    weight_decay = l[i]

    cfg.weight_decay = weight_decay
    cfg.checkpoint_path = os.path.join(checkpoint_dir, f'general_paramater_wd_{weight_decay}.pth')

    try:
        history = train_model(model, train_loader, val_loader, cfg)
    except KeyboardInterrupt:
        history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc", "spectral")}

    val_loss, val_acc     = evaluate(model, val_loader)
    train_loss, train_acc = evaluate(model, train_loader)

    print(f"\nFinal — train acc: {train_acc:.3f}  val acc: {val_acc:.3f}")

    if cfg.checkpoint_path:
        save_checkpoint(model, history, cfg.checkpoint_path)



Epoch   100/100  train loss 1.8410 acc 0.585  val loss 8.2107 acc 0.038
Training time: 3.4s

Final — train acc: 0.605  val acc: 0.038
Checkpoint saved → ./skeletons/general_paramater_wd_0.07.pth
Epoch   100/100  train loss 0.0068 acc 1.000  val loss 18.3385 acc 0.100
Training time: 3.2s

Final — train acc: 1.000  val acc: 0.100
Checkpoint saved → ./skeletons/general_paramater_wd_0.08.pth
Epoch   100/100  train loss 0.0001 acc 1.000  val loss 26.6941 acc 0.106
Training time: 3.4s

Final — train acc: 1.000  val acc: 0.106
Checkpoint saved → ./skeletons/general_paramater_wd_0.09.pth
Epoch   100/100  train loss 0.0000 acc 1.000  val loss 28.9805 acc 0.120
Training time: 3.4s

Final — train acc: 1.000  val acc: 0.120
Checkpoint saved → ./skeletons/general_paramater_wd_1.pth
